In [1]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [2]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

Activity.csv
Demographics.csv
Labels.csv
Physiology.csv
Sleep.csv


In [3]:
sleep_df1 = gen_date_col(data_dict['Sleep'],tcol='timestamp')
sleep_df1

,patient_id,timestamp,state,heart_rate,respiratory_rate,snoring,date
0,0f352,2019-06-25 22:53:00,AWAKE,69.0,14.0,False,2019-06-25
1,0f352,2019-06-25 22:54:00,AWAKE,66.0,14.0,False,2019-06-25
2,0f352,2019-06-25 22:55:00,AWAKE,70.0,14.0,False,2019-06-25
3,0f352,2019-06-25 22:56:00,AWAKE,70.0,13.0,False,2019-06-25
4,0f352,2019-06-25 22:57:00,AWAKE,68.0,13.0,False,2019-06-25
...,...,...,...,...,...,...,...
461418,f220c,2019-06-30 10:47:00,AWAKE,76.0,20.0,False,2019-06-30
461419,f220c,2019-06-30 10:48:00,AWAKE,73.0,21.0,False,2019-06-30
461420,f220c,2019-06-30 10:49:00,AWAKE,65.0,18.0,False,2019-06-30
461421,f220c,2019-06-30 10:50:00,AWAKE,75.0,15.0,False,2019-06-30


In [4]:
sleep_id_array = sleep_df1.patient_id.unique()
sleep_id_array

['0f352', '16f4b', '1fbe4', '30a32', '55cd4', ..., 'c8574', 'd7a46', 'e2472', 'ec812', 'f220c']
Length: 17
Categories (17, object): ['0f352', '16f4b', '1fbe4', '30a32', ..., 'd7a46', 'e2472', 'ec812', 'f220c']

# calculate bout number and durations (any sleep before awake would be a bout)

In [46]:
import pandas as pd
import datetime
import statistics

summary_df_list =[]

for id_pick in sleep_id_array:
    print(id_pick)
    sleep_df_idPick = sleep_df1[sleep_df1['patient_id'] == id_pick] 
    
    date_array = sleep_df_idPick.date.unique()

    date_list = []
    bout_number_list = []
    bout_duration_list = []
    bout_duration_avg_list = []
    bout_duration_med_list = []
    
    for target_date in date_array:
        date_list.append(target_date)
        # in each day
        filtered_df = sleep_df_idPick[sleep_df_idPick['date'] == target_date]
        
        # Assuming your dataframe is called sleep_df_idPick
        # First, make sure timestamp is a datetime object
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
        # Create a new column based on conditions
        filtered_df['state_simple'] = filtered_df['state'].apply(lambda x: 'sleep' if x in ['DEEP', 'LIGHT', 'REM'] else 'awake')
        
        
    ##################### calculate sleep duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['state_change'] = (filtered_df['state_simple'] != filtered_df['state_simple'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['state_change', 'state_simple']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
        
        bout_number = chunk_durations['state_simple'].value_counts().get('sleep', 0)
        bout_duration = chunk_durations[chunk_durations['state_simple'] == 'sleep']['duration_min'].tolist()
        if bout_number > 0:             
            bout_duration_avg = sum(bout_duration)/bout_number
            bout_duration_med = statistics.median(bout_duration)
        else:
            bout_duration_avg = 0
            bout_duration_med = 0
        
        bout_number_list.append(bout_number)
        bout_duration_list.append(bout_duration)        
        bout_duration_avg_list.append(bout_duration_avg)        
        bout_duration_med_list.append(bout_duration_med) 
        
    sleep_summary = {
    'patient_id': id_pick,
    'date': date_list,
    'bout_number': bout_number_list,
    'bout_duration': bout_duration_list,
    'bout_duration_avg': bout_duration_avg_list,
    'bout_duration_med': bout_duration_med_list

    }

    sleep_summary_df = pd.DataFrame(sleep_summary)  
    summary_df_list.append(sleep_summary_df)


0f352
16f4b
1fbe4
30a32
55cd4
76230
93c14
96adf
a2849
b0455
c55f8
c5785
c8574
d7a46
e2472
ec812
f220c


In [47]:
final_df = pd.concat(summary_df_list, ignore_index=True)

final_df.to_csv('output/processed_data/sleep_bouts_boutDuration_newDefine.csv', index=False)
final_df

,patient_id,date,bout_number,bout_duration,bout_duration_avg,bout_duration_med
0,0f352,2019-06-25,2,"[20.0, 0.0]",10.000000,10.0
1,0f352,2019-06-26,4,"[74.0, 14.0, 98.0, 6.0]",48.000000,44.0
2,0f352,2019-06-27,6,"[34.0, 110.0, 43.0, 9.0, 36.0, 13.0]",40.833333,35.0
3,0f352,2019-06-28,7,"[50.0, 65.0, 5.0, 10.0, 11.0, 45.0, 88.0]",39.142857,45.0
4,0f352,2019-06-29,8,"[22.0, 125.0, 69.0, 23.0, 70.0, 11.0, 36.0, 12.0]",46.000000,29.5
...,...,...,...,...,...,...
830,f220c,2019-06-24,13,"[44.0, 5.0, 11.0, 7.0, 15.0, 8.0, 18.0, 5.0, 1...",21.461538,15.0
831,f220c,2019-06-25,9,"[26.0, 40.0, 55.0, 21.0, 6.0, 24.0, 19.0, 31.0...",27.333333,24.0
832,f220c,2019-06-26,4,"[6.0, 15.0, 25.0, 10.0]",14.000000,12.5
833,f220c,2019-06-29,2,"[15.0, 5.0]",10.000000,10.0
